In [0]:

# 1. CONFIGURACIÓN Y PARÁMETROS
dbutils.widgets.text("storage_name", "datalaketpint1")
v_storage = dbutils.widgets.get("storage_name")

# Configuración de acceso al Storage
spark.conf.set(
    f"fs.azure.account.key.{v_storage}.dfs.core.windows.net",
    "AZURE_STORAGE_KEY_AQUI"
)

In [0]:

from pyspark.sql.functions import col

# 2. PROCESAR VENTAS (CSV)
path_ventas = f"abfss://bronze@{v_storage}.dfs.core.windows.net/ventas.csv"

df_ventas = spark.read.format("csv") \
    .option("header", "True") \
    .option("inferSchema", "True") \
    .load(path_ventas)

# Limpieza de Ventas
df_ventas_ok = df_ventas.filter(
    (col("ID_Venta").isNotNull()) & (col("Cantidad") > 0)
)

# Guardar Ventas en Silver
df_ventas_ok.write.format("delta").mode("overwrite").save(f"abfss://silver@{v_storage}.dfs.core.windows.net/ventas_limpias")

In [0]:


# 3. PROCESAR PRODUCTOS (JSON)

path_productos_json = f"abfss://bronze@{v_storage}.dfs.core.windows.net/productos.json"

# 1. Leer el JSON
df_productos = spark.read.format("json").load(path_productos_json)

# 2. Limpieza de Productos
df_productos_ok = df_productos.filter(col("id").isNotNull())

# 3. ELIMINAR LA CARPETA SILVER SI EXISTE (Esto evita el error de formato incompatible)
path_prod_silver = f"abfss://silver@{v_storage}.dfs.core.windows.net/productos_limpios"
dbutils.fs.rm(path_prod_silver, recurse=True) 

# 4. Guardar Productos en Silver ahora sí como Delta puro
df_productos_ok.write.format("delta").mode("overwrite").save(path_prod_silver)

print("¡Capa Silver de Productos creada desde cero exitosamente!")

¡Capa Silver de Productos creada desde cero exitosamente!


In [0]:

# 4. RESULTADOS
print("¡Capa Silver procesada!")
print(f"Ventas leídas de CSV y Productos leídos de JSON.")
display(df_productos_ok.limit(5))

¡Capa Silver procesada!
Ventas leídas de CSV y Productos leídos de JSON.


category,description,id,image,price,rating,title
men's clothing,"Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday",1,https://fakestoreapi.com/img/81fPKd-2AYL._AC_SL1500_t.png,109.95,"List(120, 3.9)","Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops"
men's clothing,"Slim-fitting style, contrast raglan long sleeve, three-button henley placket, light weight & soft fabric for breathable and comfortable wearing. And Solid stitched shirts with round neck made for durability and a great fit for casual fashion wear and diehard baseball fans. The Henley style round neckline includes a three-button placket.",2,https://fakestoreapi.com/img/71-3HjGNDUL._AC_SY879._SX._UX._SY._UY_t.png,22.3,"List(259, 4.1)",Mens Casual Premium Slim Fit T-Shirts
men's clothing,"great outerwear jackets for Spring/Autumn/Winter, suitable for many occasions, such as working, hiking, camping, mountain/rock climbing, cycling, traveling or other outdoors. Good gift choice for you or your family member. A warm hearted love to Father, husband or son in this thanksgiving or Christmas Day.",3,https://fakestoreapi.com/img/71li-ujtlUL._AC_UX679_t.png,55.99,"List(500, 4.7)",Mens Cotton Jacket
men's clothing,"The color could be slightly different between on the screen and in practice. / Please note that body builds vary by person, therefore, detailed size information should be reviewed below on the product description.",4,https://fakestoreapi.com/img/71YXzeOuslL._AC_UY879_t.png,15.99,"List(430, 2.1)",Mens Casual Slim Fit
jewelery,"From our Legends Collection, the Naga was inspired by the mythical water dragon that protects the ocean's pearl. Wear facing inward to be bestowed with love and abundance, or outward for protection.",5,https://fakestoreapi.com/img/71pWzhdJNwL._AC_UL640_QL65_ML3_t.png,695.0,"List(400, 4.6)",John Hardy Women's Legends Naga Gold & Silver Dragon Station Chain Bracelet
